In [1]:
import pandas as pd
from pathlib import Path
import os
import numpy as np
from geopy.distance import geodesic
import matplotlib.pyplot as plt
import seaborn as sns
from tabulate import tabulate
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
from lifelines import WeibullFitter
import math
import numpy as np
import pandas as pd
import folium
import matplotlib.pyplot as plt
from scipy.interpolate import CubicSpline
import datetime
from pathlib import Path
import os
import pandas as pd
from tabulate import tabulate


In [2]:
category_map = {
    # --- MECHANICAL / WEAR BASED ---
    'Bremse(r)': 'Braking System', 
    'Brake(s) need adjustment': 'Braking System',
    
    'Gir': 'Drivetrain', 'Belte / Belt': 'Drivetrain', 
    'Pedaler': 'Drivetrain', 'Cranck bearing/bottom bracket ': 'Drivetrain',
    
    'Hjul': 'Wheels & Tires', 'Lite luft': 'Wheels & Tires',
    
    'Styre': 'Steering & Chassis', 'Styrelager': 'Steering & Chassis', 'Frame': 'Steering & Chassis',
    
    'Sete': 'Body & Accessories', 'Setepinneklemme': 'Body & Accessories', 
    'Støtte': 'Body & Accessories', 'Skjerm(er)': 'Body & Accessories', 
    'Ringeklokke': 'Body & Accessories', 'Basket': 'Body & Accessories',
    
    # --- ELECTRONIC/SYSTEM ---
     'Console': 'Electronics', 
    'GPS': 'Electronics', 'Battery': 'Electronics', 'Lys': 'Electronics',
    
    'Lock & unlock': 'Locking System', 'Lås': 'Locking System',
    
    # --- LOGIC & EXTERNAL (Exclude from odometer analysis) ---
    'Too many quick returns': 'Too many quick returns',
    'Unauthorized Trip': 'Unauthorized Trip',
    'Unresponsive Controller': 'Unresponsive Controller',
    'Vandalism': 'Vandalism'
}

In [3]:

# ----------------------------
# Directories setup
# ----------------------------
notebook_dir = Path(os.getcwd())
trip_base_dir = notebook_dir / "trip_data" / "raw_files"
trip_year_folders = sorted(trip_base_dir.glob("trondheimbysykkel-*"))
raw_maintenance_files_dir = notebook_dir / "maintenance_data" / "raw_files"
_maintenance_files = sorted(raw_maintenance_files_dir.glob("*.csv"))

# ----------------------------
# Load Trip Data by Year
# ----------------------------
trip_data_list = []

for folder in trip_year_folders:
    parquet_files = sorted(folder.glob("*.parquet"))
    
    # Load and collect dataframes
    if parquet_files:
        year_dfs = [pd.read_parquet(file) for file in parquet_files]
        trip_data_list.append(pd.concat(year_dfs, ignore_index=True))

# ----------------------------
# Combine into master trip_data
# ----------------------------
trip_data = pd.concat(trip_data_list, ignore_index=True) if trip_data_list else pd.DataFrame()

# ----------------------------
# Single Clean Summary
# ----------------------------
final_summary = [
    ["Trip year folders", len(trip_year_folders)],
    ["Maintenance files", len(_maintenance_files)],
    ["Total Rows", f"{len(trip_data):,}"],
    ["Total Columns", trip_data.shape[1]],
]

print("\nLOAD COMPLETE")
print(tabulate(final_summary, headers=["Metric", "Value"], tablefmt="github"))


LOAD COMPLETE
| Metric            | Value      |
|-------------------|------------|
| Trip year folders | 3          |
| Maintenance files | 3          |
| Total Rows        | 19,339,524 |
| Total Columns     | 24         |


In [4]:
# ---------------------------------------------------------
# Load Maintenance Data
# ---------------------------------------------------------
maint_store = {'damage': [], 'repair': [], 'maintenance': []}

for file in _maintenance_files:
    df = pd.read_csv(file)
    fname = file.name.lower()
    
    # Sort into correct bucket based on filename
    category = 'damage' if 'damage' in fname else 'repair' if 'repair' in fname else 'maintenance'
    maint_store[category].append(df)

# ---------------------------------------------------------
# Create DataFrames dynamically
# ---------------------------------------------------------
damage_data = pd.concat(maint_store['damage'], ignore_index=True) if maint_store['damage'] else pd.DataFrame()
repair_data = pd.concat(maint_store['repair'], ignore_index=True) if maint_store['repair'] else pd.DataFrame()
maintenance_data = pd.concat(maint_store['maintenance'], ignore_index=True) if maint_store['maintenance'] else pd.DataFrame()

# ---------------------------------------------------------
# Concise, pretty summary
# ---------------------------------------------------------
stats = [
    ["Damage Logs", f"{len(damage_data):,}"],
    ["Repair Logs", f"{len(repair_data):,}"],
    ["Maint Logs",  f"{len(maintenance_data):,}"]
]

print("\n" + "="*30)
print(tabulate(stats, headers=["Dataset", "Rows"], tablefmt="github"))
print("="*30)


| Dataset     | Rows    |
|-------------|---------|
| Damage Logs | 756,565 |
| Repair Logs | 250,815 |
| Maint Logs  | 363,324 |


In [5]:
# ---------------------------------------------------------------
# Configuration: (DataFrame, Date Columns, Sort Column)
# ---------------------------------------------------------------
cleaning_map = [
    (trip_data,        ['position_at', 'trip_started_at', 'trip_ended_at'], 'trip_started_at'),
    (damage_data,      ['created_at', 'resolved_at'],                      'created_at'),
    (maintenance_data, ['started_at', 'completed_at'],                     'completed_at'),
    (repair_data,      ['created_at', 'started_at', 'completed_at'],       'completed_at')
]

# ---------------------------------------------------------------
# Process each DataFrame according to the map
# ---------------------------------------------------------------
for df, date_cols, sort_col in cleaning_map:
    if df is not None and not df.empty:
        # Convert Dates: timezone-naive nanosecond precision
        for col in date_cols:
            if col in df.columns:
                df[col] = pd.to_datetime(df[col], errors='coerce').dt.tz_localize(None).astype('datetime64[ns]')
        
        # Optimize IDs: Find 'id' columns and convert to Int64
        id_cols = [c for c in df.columns if 'id' in c.lower()]
        for c in id_cols:
            df[c] = pd.to_numeric(df[c], errors='coerce').astype('Int64')
            
        # Sort for merge_asof readiness
        df.sort_values(['vehicle_id', sort_col], inplace=True)

# ---------------------------------------------------------------
# Final concise summary of datasets
# ---------------------------------------------------------------
type_stats = [
    ["Trips", f"{trip_data.shape[0]:,}", "trip_started_at"],
    ["Damage", f"{damage_data.shape[0]:,}", "created_at"],
    ["Maint", f"{maintenance_data.shape[0]:,}", "completed_at"],
    ["Repair", f"{repair_data.shape[0]:,}", "completed_at"]
]

print("\n" + "="*50)
print(tabulate(type_stats, headers=["Dataset", "Rows", "Sorted By"], tablefmt="github"))
print("="*50)
print("SUCCESS: All time columns localized and ID columns optimized.")


| Dataset   | Rows       | Sorted By       |
|-----------|------------|-----------------|
| Trips     | 19,339,524 | trip_started_at |
| Damage    | 756,565    | created_at      |
| Maint     | 363,324    | completed_at    |
| Repair    | 250,815    | completed_at    |
SUCCESS: All time columns localized and ID columns optimized.


In [6]:
# ---------------------------------------------------------------
# define target case and years for filtering
# ---------------------------------------------------------------
TARGET_CASE = 'trondheim'
TARGET_YEARS = [2024, 2023] 

# ---------------------------------------------------------------
# Using a dictionary to explicitly update the variables and prevent column loss
# ---------------------------------------------------------------
to_filter = {
    'damage_data':      (damage_data,      'created_at',    'Damage Reports'),
    'maintenance_data': (maintenance_data, 'completed_at',  'Maintenance Jobs'),
    #'repair_data':      (repair_data,      'completed_at',  'Repairs Logged'),
    'trip_data':        (trip_data,        'trip_ended_at', 'Total Trips')
}

filtered_results = []

# ---------------------------------------------------------------
# Loop through each dataset, apply the filter, and update the global variables
# ---------------------------------------------------------------
for var_name, (df, date_col, label) in to_filter.items():
    if df is not None and not df.empty:
        mask = df[date_col].dt.year.isin(TARGET_YEARS)
        if 'case' in df.columns:
            mask &= (df['case'] == TARGET_CASE)
            
        # Explicit copy to preserve ALL columns (including damage_id)
        filtered_df = df[mask].copy()
        globals()[var_name] = filtered_df
        filtered_results.append([label, f"{len(filtered_df):,}"])

# ---------------------------------------------------------------
# Print Year/Case Status
# ---------------------------------------------------------------
years_str = ", ".join(map(str, TARGET_YEARS))
print(f"\n{'='*45}\n{TARGET_CASE.upper()} FLEET STATUS ({years_str})\n{'='*45}")
print(tabulate(filtered_results, headers=["Dataset", "Count"], tablefmt="github"))


TRONDHEIM FLEET STATUS (2024, 2023)
| Dataset          | Count     |
|------------------|-----------|
| Damage Reports   | 65,202    |
| Maintenance Jobs | 38,684    |
| Total Trips      | 9,620,493 |


In [7]:
# drop columns in trip_data that are not needed
trip_data = trip_data.drop(columns=['position_wkt', 'trip_unlock_method', 'trip_end_dock_number', 'trip_start_dock_number', 'product_id', 'product_name', 'product_price', 'product_sales_channel', 'product_sales_locale', 'sale_from_value_code', 'user_id', 'trip_state'])

In [8]:
# ---------------------------------------------------------------
# Print NULL values whole trip_data
# ---------------------------------------------------------------
null_start = trip_data['trip_start_dock_group_title'].isnull().sum()
null_end = trip_data['trip_end_dock_group_title'].isnull().sum()
print(f"- trip_start_dock_group_title: {null_start:,} NULLs")
print(f"- trip_end_dock_group_title: {null_end:,} NULLs")

# ---------------------------------------------------------------
# Remove rows with NULLs in critical columns (if any)
# ---------------------------------------------------------------
trip_data = trip_data.dropna(subset=['trip_start_dock_group_title', 'trip_end_dock_group_title'])

- trip_start_dock_group_title: 0 NULLs
- trip_end_dock_group_title: 265,698 NULLs


In [9]:
# ---------------------------------------------------------------
# Pre-process and Identify
# ---------------------------------------------------------------
damage_data['created_hour'] = damage_data['created_at'].dt.hour
initial_count = len(damage_data)

# ---------------------------------------------------------------
# Define criteria for automated system noise
# ---------------------------------------------------------------
noise_types = [
    'Unresponsive Controller'     # Connectivity/Firmware (The big one)
]
cutoff_time = datetime.time(6, 0, 0)

# ---------------------------------------------------------------
# get the maintenance id connected to the damage data
# ---------------------------------------------------------------
#connected_maint_id = damage_data['maintenance_id'].notna()

# ---------------------------------------------------------------
# if the damage data have an maintenance id. and the corresponding maintenance has a comment with "null (Autoresolved)"
# ---------------------------------------------------------------
#maint_comment = maintenance_data.set_index('maintenance_id')['comment']
#damage_data['maint_comment'] = damage_data['maintenance_id'].map(maint_comment)


is_noise = (damage_data['damage_type_name'].isin(noise_types)) & \
           (damage_data['created_at'].dt.time < cutoff_time)

# ---------------------------------------------------------------
# Filter
# --------------------------------------------------------------- 
removed_rows = damage_data[is_noise].copy()
damage_data = damage_data[~is_noise].copy()

# ---------------------------------------------------------------
# Print summary
# ---------------------------------------------------------------
cleanup_stats = [
    ["Initial Logs", f"{initial_count:,}"],
    ["System Noise Removed", f"{len(removed_rows):,}"],
    ["Cleaned Dataset", f"{len(damage_data):,}"]
]

print(f"\n{'='*40}\nNOISE CLEANUP: OVERNIGHT SYSTEM FLAGS\n{'='*40}")
print(tabulate(cleanup_stats, headers=["Metric", "Count"], tablefmt="github"))

if not removed_rows.empty:
    print(f"\nSAMPLE OF REMOVED NOISE (PRE-06:01):")
    print(removed_rows[['created_at', 'damage_type_name']].to_string(index=False))
print("="*40)


NOISE CLEANUP: OVERNIGHT SYSTEM FLAGS
| Metric               | Count   |
|----------------------|---------|
| Initial Logs         | 65,202  |
| System Noise Removed | 38,954  |
| Cleaned Dataset      | 26,248  |

SAMPLE OF REMOVED NOISE (PRE-06:01):
             created_at        damage_type_name
2023-06-20 05:00:03.696 Unresponsive Controller
2023-07-02 05:00:03.178 Unresponsive Controller
2023-07-04 05:00:03.129 Unresponsive Controller
2023-07-20 05:00:02.981 Unresponsive Controller
2023-07-21 05:00:02.070 Unresponsive Controller
2023-07-23 05:00:02.011 Unresponsive Controller
2023-07-28 05:00:02.673 Unresponsive Controller
2023-08-04 05:00:02.129 Unresponsive Controller
2023-08-05 05:00:02.128 Unresponsive Controller
2023-08-06 05:00:02.064 Unresponsive Controller
2023-08-07 05:00:02.619 Unresponsive Controller
2023-08-08 05:00:02.284 Unresponsive Controller
2023-08-09 05:00:02.078 Unresponsive Controller
2023-08-10 05:00:02.991 Unresponsive Controller
2023-08-11 05:00:02.656 Unre

In [10]:
def haversine_distance(lat1, lon1, lat2, lon2):
    # Earth's radius in kilometers
    r = 6371 
    
    # Convert degrees to radians
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    
    # Apply formula
    a = np.sin(dphi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda/2)**2
    c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
    
    return r * c # Returns distance in km

In [11]:
# ---------------------------------------------------------------
# Stationary Ping Cleanup
# ----------------------------------------------------------------

# ---------------------------------------------------------------
# Sort and Shift
# ---------------------------------------------------------------
trip_data = trip_data.sort_values(['vehicle_id', 'position_at']).reset_index(drop=True)
grp = trip_data.groupby('vehicle_id')

trip_data['prev_time'] = grp['position_at'].shift(1)
trip_data['prev_trip_id'] = grp['trip_id'].shift(1)

# ---------------------------------------------------------------
# Distance Calculation (meters)
# ---------------------------------------------------------------
trip_data['dist_from_prev_m'] = haversine_distance(
    trip_data['position_latitude'], trip_data['position_longitude'],
    grp['position_latitude'].shift(1), grp['position_longitude'].shift(1)
) * 1000

# ---------------------------------------------------------------
# Identify and Remove
# Threshold set to 0.0 to catch only absolute duplicates; adjust to 5.0 for GPS jitter
# ---------------------------------------------------------------
GPS_NOISE_THRESHOLD_M = 0.0
is_stationary = (trip_data['trip_id'] == trip_data['prev_trip_id']) & \
                (trip_data['dist_from_prev_m'] <= GPS_NOISE_THRESHOLD_M)

rows_before = len(trip_data)
stat_rows = trip_data[is_stationary].copy()
trip_data = trip_data[~is_stationary].copy()

# ---------------------------------------------------------------
# Summary
# ---------------------------------------------------------------
diffs = (stat_rows['position_at'] - stat_rows['prev_time']).dt.total_seconds()

stats = [
    ["Rows Before", f"{rows_before:,}"],
    ["Stationary Removed", f"{len(stat_rows):,}"],
    ["Rows After", f"{len(trip_data):,}"],
    ["Reduction %", f"{(1 - len(trip_data)/rows_before):.1%}"]
]

time_stats = [
    ["Mean Gap", f"{diffs.mean():.1f}s"],
    ["Median Gap", f"{diffs.median():.1f}s"],
    ["Max Gap", f"{diffs.max():.1f}s"]
]

print(f"\n{'='*35}\nSTATIONARY PING REMOVAL\n{'='*35}")
print(tabulate(stats, tablefmt="plain"))
print(f"\nTime Stats for Removed Pings:")
print(tabulate(time_stats, tablefmt="plain"))

if not stat_rows.empty:
    print(f"\nSample Data (First 5):")
    sample = stat_rows[['trip_id', 'position_at', 'dist_from_prev_m']].head(5)
    print(tabulate(sample, headers='keys', tablefmt="simple", showindex=False))
print("="*35)


STATIONARY PING REMOVAL
Rows Before         9,354,795
Stationary Removed  1,546,751
Rows After          7,808,044
Reduction %         16.5%

Time Stats for Removed Pings:
Mean Gap    14.0s
Median Gap  15.3s
Max Gap     7319.3s

Sample Data (First 5):
  trip_id  position_at                   dist_from_prev_m
---------  --------------------------  ------------------
 54865005  2023-04-16 14:24:16.462000                   0
 54865005  2023-04-16 14:24:20.104000                   0
 54865005  2023-04-16 14:25:01.446000                   0
 54865005  2023-04-16 14:25:16.776000                   0
 54865005  2023-04-16 14:36:02.328000                   0


In [12]:
# ---------------------------------------------------------------
# Apply Mapping
# ---------------------------------------------------------------
damage_data['damage_category'] = damage_data['damage_type_name'].map(category_map).fillna('Other')

# ---------------------------------------------------------------
# Filter Exclusions
# ---------------------------------------------------------------
exclude_list = ['Too many quick returns', 'Unauthorized Trip', 'Unresponsive Controller', 'Vandalism', 'electronics', 'Locking System']
initial_count = len(damage_data)
damage_data = damage_data[~damage_data['damage_category'].isin(exclude_list)].copy()

# ---------------------------------------------------------------
# Summary
# ---------------------------------------------------------------
stats = [
    ["Initial Reports", f"{initial_count:,}"],
    ["Excluded (System/Logic)", f"{initial_count - len(damage_data):,}"],
    ["Cleaned Dataset", f"{len(damage_data):,}"]
]

print(f"\n{'='*35}\nCATEGORY FILTERING SUMMARY\n{'='*35}")
print(tabulate(stats, tablefmt="plain"))
print("="*35)


CATEGORY FILTERING SUMMARY
Initial Reports          26,248
Excluded (System/Logic)  13,949
Cleaned Dataset          12,299


In [13]:
# ---------------------------------------------------------------
# Deduplication Logic
# -> Multiple reports of SAME category in SAME maintenance = 1 event
# ---------------------------------------------------------------
damage_data = damage_data.sort_values(
    ['vehicle_id', 'asset_maintenance_id', 'damage_category', 'created_at']
)

# Keep first report of each (vehicle, maintenance_id, category) combination
damage_data_unique = damage_data.groupby(
    ['vehicle_id', 'asset_maintenance_id', 'damage_category']
).agg({
    'created_at': 'first',           # Time of first report
    'damage_type_name': 'first',     # Specific damage name
    'damage_id': 'count',            # Use damage_id to count reports
    'created_hour': 'first'          # Keep the hour
}).reset_index()

damage_data_unique.rename(columns={'damage_id': 'report_count'}, inplace=True)

# ---------------------------------------------------------------
# Statistics Summary
# ---------------------------------------------------------------
total_reports = len(damage_data)
unique_events = len(damage_data_unique)
repeat_reports = total_reports - unique_events

stats = [
    ["Original Reports", f"{total_reports:,}"],
    ["Unique Damage Events", f"{unique_events:,}"],
    ["Duplicates Removed", f"{repeat_reports:,}"],
    ["Reduction %", f"{(repeat_reports/total_reports):.1%}"]
]

print(f"\n{'='*40}\nDEDUPLICATION RESULTS\n{'='*40}")
print(tabulate(stats, tablefmt="plain"))

'''# --- 3. Example Comparison ---
example_maint_id = 396574

# Check if the specific example exists in the filtered set
if example_maint_id in damage_data['asset_maintenance_id'].values:
    print(f"\nMAINTENANCE SESSION EXAMPLE: {example_maint_id}")

    # Helper for mini-tables
    def fmt_mini(df, cols, headers):
        temp = df[df['asset_maintenance_id'] == example_maint_id].copy()
        temp['time'] = temp['created_at'].dt.strftime('%H:%M:%S')
        return tabulate(temp[['time'] + cols], headers=['Time'] + headers, tablefmt="simple", showindex=False)

    print(f"\nBEFORE (Original Logs):")
    print(fmt_mini(damage_data, ['damage_category', 'damage_type_name'], ['Category', 'Type']))

    print(f"\nAFTER (Deduplicated):")
    print(fmt_mini(damage_data_unique, ['damage_category', 'damage_type_name', 'report_count'], ['Category', 'Type', '# Reports']))

print("="*40)'''

# Finalize
damage_data = damage_data_unique


DEDUPLICATION RESULTS
Original Reports      12,299
Unique Damage Events  7,673
Duplicates Removed    4,626
Reduction %           37.6%


In [14]:
# ---------------------------------------------------------------
# Sort to ensure chronological order per vehicle
# ---------------------------------------------------------------
trip_data = trip_data.sort_values(['vehicle_id', 'position_at']).reset_index(drop=True)

# ---------------------------------------------------------------
# Create 'Previous' columns using shift
# This puts the previous row's data on the current row for easy comparison
# ---------------------------------------------------------------
grp = trip_data.groupby('vehicle_id')

trip_data['prev_end_station_id'] = grp['trip_end_dock_group_id'].shift(1)
trip_data['prev_end_station_name'] = grp['trip_end_dock_group_title'].shift(1)
trip_data['prev_lat'] = grp['position_latitude'].shift(1)
trip_data['prev_lon'] = grp['position_longitude'].shift(1)
trip_data['prev_time'] = grp['position_at'].shift(1)
trip_data['prev_trip_id'] = grp['trip_id'].shift(1)

In [15]:
# ---------------------------------------------------------------
# Calculate distance and time
# ---------------------------------------------------------------
trip_data['dist_km'] = haversine_distance(
    trip_data['position_latitude'], trip_data['position_longitude'],
    trip_data['prev_lat'], trip_data['prev_lon']
)

trip_data['time_diff_hr'] = (trip_data['position_at'] - trip_data['prev_time']).dt.total_seconds() / 3600
trip_data['speed_kmh'] = trip_data['dist_km'] / trip_data['time_diff_hr'].replace(0, np.nan)

In [16]:
# ---------------------------------------------------------------
# Initialize movement_type as "Valid" by default
# ---------------------------------------------------------------
trip_data['movement_type'] = "Valid"

# ---------------------------------------------------------------
# Boolean Masks for Movement Classification
# ---------------------------------------------------------------
is_same_trip   = trip_data['trip_id'] == trip_data['prev_trip_id']
is_new_trip    = (trip_data['trip_id'] != trip_data['prev_trip_id']) & trip_data['prev_trip_id'].notna()
is_too_fast    = trip_data['speed_kmh'] > 30
is_moved_far   = trip_data['dist_km'] > 0.15 
is_extreme_rebalancing = trip_data['dist_km'] >= 10.0

# ---------------------------------------------------------------
# Station Logic 
# ---------------------------------------------------------------

# 1. Did the start station change WITHIN the same trip? (Data Error)
start_station_changed = trip_data['trip_start_dock_group_id'] != trip_data.groupby('trip_id')['trip_start_dock_group_id'].transform('first')

# 2. Does the current start match the previous end? (For Inter-trip)
stations_match = trip_data['trip_start_dock_group_id'] == trip_data['prev_end_station_id']

# ---------------------------------------------------------------
# APPLY RULES
# ---------------------------------------------------------------

# RULE A: Intra-trip Station Consistency (CRITICAL)
# If the metadata changes mid-trip, flag it regardless of speed.
trip_data.loc[is_same_trip & start_station_changed, 'movement_type'] = "Error: Mid-Trip Station Change"

# RULE B: Intra-trip Teleportation
trip_data.loc[is_same_trip & is_too_fast & ~start_station_changed, 'movement_type'] = "Invalid: Speed"

# RULE C: Logistical Rebalance (The "Not Valid" one to remove)
# If stations don't match AND it moved more than 10km, it's a van move.
is_logistical_move = is_new_trip & (~stations_match) & is_extreme_rebalancing
trip_data.loc[is_logistical_move, 'movement_type'] = "Invalid: Van Rebalance"

# RULE D: Valid Rebalance (The "Short" one to keep)
# If stations don't match but it moved LESS than 10km, we count it as valid wear-and-tear.
is_short_move = is_new_trip & (~stations_match) & (~is_extreme_rebalancing) & is_moved_far
trip_data.loc[is_short_move, 'movement_type'] = "Valid" # We keep this as "Valid"

# RULE E: Unexplained Gap
trip_data.loc[is_new_trip & is_moved_far & stations_match, 'movement_type'] = "Unexplained Move"

In [17]:
# ---------------------------------------------------------------
# Filter Criteria
# ---------------------------------------------------------------
exclude_list = ["Unexplained Move", "GPS noise", "Invalid: Speed", "Invalid: Van Rebalance"]

# ---------------------------------------------------------------
# Calculate impact before dropping
# ---------------------------------------------------------------
removed_dist = (
    trip_data[trip_data['movement_type'].isin(exclude_list)]
    .groupby('vehicle_id')['dist_km']
    .sum()
    .reset_index(name='removed_km')
)


# ---------------------------------------------------------------
# Identify extreme outliers (>10km) within the invalid set for verification
# ---------------------------------------------------------------
extreme_errors = trip_data[
    (trip_data['movement_type'] == "Invalid: Van Rebalance") & (trip_data['dist_km'] > 10)
]

# ---------------------------------------------------------------
# Apply Filters
# ---------------------------------------------------------------
initial_rows = len(trip_data)
trip_data = trip_data[~trip_data['movement_type'].isin(exclude_list)].copy()
trip_data_clean = trip_data # Ensuring consistency across references

# ---------------------------------------------------------------
# Summary
# ---------------------------------------------------------------
filter_stats = [
    ["Initial Pings", f"{initial_rows:,}"],
    ["Removed Pings", f"{initial_rows - len(trip_data):,}"],
    ["Total KM Excised", f"{removed_dist['removed_km'].sum():,.1f} km"],
    ["Cleaned Dataset", f"{len(trip_data):,}"]
]

print(f"\n{'='*45}\nMOVEMENT FILTERING: REMOVING SYSTEM NOISE\n{'='*45}")
print(tabulate(filter_stats, tablefmt="plain"))

if not extreme_errors.empty:
    print(f"\nEXTREME TELEPORTATION DETECTED (>10km):")
    print(tabulate(
        extreme_errors[['vehicle_id', 'position_at', 'dist_km']].head(5),
        headers=['Vehicle ID', 'Time', 'Distance (km)'],
        tablefmt="simple",
        showindex=False,
        floatfmt=".2f"
    ))
print("="*45)


MOVEMENT FILTERING: REMOVING SYSTEM NOISE
Initial Pings     7,808,044
Removed Pings     172,317
Total KM Excised  28,710,167.7 km
Cleaned Dataset   7,635,727

EXTREME TELEPORTATION DETECTED (>10km):
  Vehicle ID  Time                          Distance (km)
------------  --------------------------  ---------------
          20  2024-04-19 12:56:33.691000          6903.47
          21  2024-08-05 17:06:46.114000          7017.48
          23  2024-06-26 07:26:27.065000          7027.65
          25  2023-06-12 10:30:00.803000           492.77
          27  2023-05-01 10:52:02.749000          6968.72


In [18]:

# ---------------------------------------------------------------
# Calculate valid accumulated km per bike
# ---------------------------------------------------------------
accumulated_km = (
    trip_data_clean.groupby('vehicle_id')['dist_km']
    .sum()
    .reset_index()
    .rename(columns={'dist_km': 'clean_km'})
)

# ---------------------------------------------------------------
# Merge for the final audit report
# ---------------------------------------------------------------
audit_report = pd.merge(accumulated_km, removed_dist, on='vehicle_id', how='outer').fillna(0)

# Calculate totals and percentage of distance lost to rebalancing/noise
audit_report['total_raw_km'] = audit_report['clean_km'] + audit_report['removed_km']
audit_report['%_removed'] = (audit_report['removed_km'] / audit_report['total_raw_km']) * 100

audit_report = audit_report.sort_values(by='clean_km', ascending=False)

print(f"\n=== ACCUMULATED KM BY VEHICLE (WITH LOGISTICS AUDIT) ===")
print(tabulate(
    audit_report, 
    headers=['Vehicle ID', 'Riding KM (Clean)', 'Removed KM', 'Raw Total', '% Removed'], 
    tablefmt="psql", 
    showindex=False,
    floatfmt=".2f"
))

# export the final trip data, damage data and maintenance data for the next steps of the analysis
trip_data_clean.to_csv(notebook_dir / "trip_data" / "processed" / "trip_data_clean.csv", index=False)
damage_data.to_csv(notebook_dir / "maintenance_data" / "processed" / "damage_data_clean.csv", index=False)
maintenance_data.to_csv(notebook_dir / "maintenance_data" / "processed" / "maintenance_data_clean.csv", index=False)



=== ACCUMULATED KM BY VEHICLE (WITH LOGISTICS AUDIT) ===
+--------------+---------------------+--------------+-------------+-------------+
|   Vehicle ID |   Riding KM (Clean) |   Removed KM |   Raw Total |   % Removed |
|--------------+---------------------+--------------+-------------+-------------|
|          384 |             3478.65 |    182676.52 |   186155.16 |       98.13 |
|         4286 |             3418.53 |    261961.78 |   265380.31 |       98.71 |
|         4018 |             3151.50 |    329957.84 |   333109.34 |       99.05 |
|           58 |             2851.57 |    214089.78 |   216941.35 |       98.69 |
|         3963 |             2713.32 |    314255.92 |   316969.24 |       99.14 |
|          233 |             2665.04 |    244747.48 |   247412.53 |       98.92 |
|           37 |             2638.34 |    142219.10 |   144857.44 |       98.18 |
|         4070 |             2613.71 |    303462.22 |   306075.93 |       99.15 |
|         1166 |             2462.15 |  